# Exploratory Data Analysis: Campus Carbon Emissions & Activity Dynamics
**Academic Project:** BDS-36 | T.Y. B.Sc. Data Science, Semester V | Academic Year 2026-27  
**Dataset:** Validated Campus Monthly Activity (`data/processed/cleaned_activity.csv`) & Carbon Accounting Output (`data/processed/campus_emissions.csv`)  
**Methodological Standard:** GHG Protocol Corporate Standard (Operational Control Approach)

---

### Epistemic & Methodological Notice:
1. **Non-Causal Framework:** All analyses herein report observed mathematical associations, seasonal patterns, and accounting distributions. **No unverified causal mechanisms are claimed**.
2. **Clear Distinction:**
   - **Observed Characteristics:** Empirical values, totals, and distributions resulting from the 36-month dataset.
   - **Underlying Parameterized Assumptions:** Academic semester calendars (vacations in June/Dec), climate cooling indices, and static documented emission factors from official sources (CEA India 2024, DEFRA 2023.1).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Publication figure aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.sans-serif'] = 'Arial'

print("Analysis environment ready.")


In [ ]:
# Load validated activity data, computed carbon emissions, and factor registry
emissions_df = pd.read_csv('../data/processed/campus_emissions.csv')
factors_df = pd.read_csv('../data/emission_factors.csv')

emissions_df['date'] = pd.to_datetime(emissions_df['date'])
emissions_df['year'] = emissions_df['date'].dt.year
emissions_df['month'] = emissions_df['date'].dt.month
emissions_df['year_month'] = emissions_df['date'].dt.strftime('%Y-%m')

print(f"Emissions Dataset Shape: {emissions_df.shape[0]} rows x {emissions_df.shape[1]} columns")
print(f"Date Range: {emissions_df['date'].min().strftime('%Y-%m-%d')} to {emissions_df['date'].max().strftime('%Y-%m-%d')}")
print(f"Facilities Tracked ({len(emissions_df['building'].unique())}): {list(emissions_df['building'].unique())}")


---
## 1. Monthly Emissions Trajectory & Seasonal Cycles

### Analysis Questions:
- How do total campus emissions evolve over the 36-month timeline?
- Are recurring cyclical troughs and peaks observable across annual billing cycles?

![Figure 1: Monthly Campus Carbon Emissions Trajectory](../reports/figures/01_monthly_emissions_trend.png)


In [ ]:
# Monthly aggregation across campus
monthly_totals = emissions_df.groupby('year_month').agg({
    'electricity_emissions_kg': 'sum',
    'travel_emissions_kg': 'sum',
    'waste_emissions_kg': 'sum',
    'procurement_emissions_kg': 'sum',
    'total_emissions_kg': 'sum'
}) / 1000.0  # Convert to tCO2e

print("Top 3 Peak Emission Months:")
print(monthly_totals['total_emissions_kg'].sort_values(ascending=False).head(3))

print("\nTop 3 Lowest Emission Months (Troughs):")
print(monthly_totals['total_emissions_kg'].sort_values(ascending=True).head(3))


> [!NOTE]
> **Observations vs. Assumptions (Monthly Trajectory):**
> - **Empirical Observations:**
>   - Campus emissions reach recurring annual peaks in **April** (~131.8 tCO2e) and **August** (~136.3 tCO2e).
>   - Pronounced drops occur consistently in **June** (~74.7 tCO2e, a ~45% reduction from peak) and **December** (~69.0 tCO2e).
> - **Underlying Parameterized Assumptions:**
>   - The observed troughs reflect the academic calendar assumptions modeled in `src/data_generator.py` (summer break in May–June with 85% student headcount drop; winter break in December).
>   - The April and August peaks coincide with higher climate cooling factors (1.30–1.45) and active semester attendance.


---
## 2. Yearly Emissions & Inter-Annual Growth

### Analysis Questions:
- What are the cumulative annual emissions for 2023, 2024, and 2025?
- What is the year-over-year rate of emissions change?

![Figure 2: Annual Campus Carbon Emissions by Category](../reports/figures/02_yearly_emissions_comparison.png)


In [ ]:
yearly_summary = emissions_df.groupby('year').agg({
    'total_emissions_kg': 'sum',
    'electricity_emissions_kg': 'sum',
    'travel_emissions_kg': 'sum',
    'waste_emissions_kg': 'sum',
    'procurement_emissions_kg': 'sum'
})
yearly_summary['total_emissions_mt'] = yearly_summary['total_emissions_kg'] / 1000.0
yearly_summary['yoy_growth_pct'] = yearly_summary['total_emissions_kg'].pct_change() * 100.0

yearly_summary[['total_emissions_kg', 'total_emissions_mt', 'yoy_growth_pct']].round(2)


> [!NOTE]
> **Observations vs. Assumptions (Annual Trends):**
> - **Empirical Observations:**
>   - 2023 total: **1,361.81 tCO2e** ($1,361,814	ext{ kg}$)
>   - 2024 total: **1,405.15 tCO2e** ($1,405,149	ext{ kg}$, +3.18% YoY)
>   - 2025 total: **1,429.23 tCO2e** ($1,429,226	ext{ kg}$, +1.71% YoY)
>   - 3-Year Total: **4,196.19 tCO2e** ($4,196,188.51	ext{ kg}$)
> - **Underlying Parameterized Assumptions:**
>   - Secular growth rates of 2.0%–2.5% per annum were parameterized in the synthetic generator to reflect organic campus expansion and digital equipment adoption.


---
## 3. Emission Category Contribution & Pareto Analysis

### Analysis Questions:
- Which activity category constitutes the primary driver of the campus carbon footprint?
- What are the proportional shares of Electricity, Commuter Travel, Solid Waste, and Procurement?

![Figure 3: Category Contribution to Total Campus Emissions](../reports/figures/03_category_contribution.png)


In [ ]:
tot_elec = emissions_df['electricity_emissions_kg'].sum()
tot_travel = emissions_df['travel_emissions_kg'].sum()
tot_waste = emissions_df['waste_emissions_kg'].sum()
tot_proc = emissions_df['procurement_emissions_kg'].sum()
grand_total = emissions_df['total_emissions_kg'].sum()

category_breakdown = pd.DataFrame([
    {'Category': 'Electricity', 'Emissions (kg)': tot_elec, 'Emissions (t)': tot_elec / 1000.0, 'Share (%)': tot_elec / grand_total * 100},
    {'Category': 'Travel', 'Emissions (kg)': tot_travel, 'Emissions (t)': tot_travel / 1000.0, 'Share (%)': tot_travel / grand_total * 100},
    {'Category': 'Waste', 'Emissions (kg)': tot_waste, 'Emissions (t)': tot_waste / 1000.0, 'Share (%)': tot_waste / grand_total * 100},
    {'Category': 'Procurement', 'Emissions (kg)': tot_proc, 'Emissions (t)': tot_proc / 1000.0, 'Share (%)': tot_proc / grand_total * 100},
]).sort_values(by='Share (%)', ascending=False)

category_breakdown.round(2)


> [!NOTE]
> **Observations vs. Assumptions (Category Breakdown):**
> - **Empirical Observations:**
>   - **Electricity is the dominant driver**, accounting for **79.77%** ($3,347.35	ext{ tCO}_2	ext{e}$) of all emissions.
>   - **Travel** is the second largest driver at **14.16%** ($594.09	ext{ tCO}_2	ext{e}$).
>   - **Waste** contributes **5.20%** ($218.03	ext{ tCO}_2	ext{e}$).
>   - **Procurement** represents **0.87%** ($36.71	ext{ tCO}_2	ext{e}$).
> - **Underlying Parameterized Assumptions:**
>   - Electricity emission factor of **0.716 kgCO2e/kWh** (CEA India CO2 Baseline Database v19, 2024 generation-end average).
>   - Commuting factor of **0.140 kgCO2e/km** (DEFRA 2023.1 passenger mix).
>   - Waste factor of **0.446 kgCO2e/kg** (DEFRA 2023.1 commercial/industrial waste).
>   - Procurement spend factor of **0.00042 kgCO2e/INR** (CEDA/ADB EEIO 2023).


---
## 4. Facility / Building Carbon Intensity Breakdown

### Analysis Questions:
- How are emissions distributed across different functional building archetypes?
- Which building represents the highest priority target for decarbonization?

![Figure 4: Cumulative Emissions Breakdown by Campus Facility](../reports/figures/04_building_contribution.png)


In [ ]:
building_summary = emissions_df.groupby('building').agg({
    'total_emissions_kg': 'sum',
    'electricity_emissions_kg': 'sum',
    'travel_emissions_kg': 'sum',
    'waste_emissions_kg': 'sum',
    'procurement_emissions_kg': 'sum',
    'student_count': 'mean',
    'staff_count': 'mean'
})
building_summary['total_emissions_t'] = building_summary['total_emissions_kg'] / 1000.0
building_summary['share_pct'] = building_summary['total_emissions_kg'] / grand_total * 100.0

building_summary[['total_emissions_t', 'share_pct', 'student_count', 'staff_count']].sort_values(by='total_emissions_t', ascending=False).round(2)


> [!NOTE]
> **Observations vs. Assumptions (Facility Distributions):**
> - **Empirical Observations:**
>   - **Science & Engineering Complex** produces the highest emissions: **1,317.28 tCO2e (31.39%)**.
>   - **Hostel & Dining Complex** ranks second: **920.98 tCO2e (21.95%)**.
>   - **Central Library & Student Hub** ranks third: **837.11 tCO2e (19.95%)**.
>   - **Academic Block - Arts & Humanities**: **649.56 tCO2e (15.48%)**.
>   - **Administrative Headquarters**: **471.25 tCO2e (11.23%)**.
> - **Underlying Parameterized Assumptions:**
>   - S&E complex profile reflects high base electrical loads (42,000 kWh/mo) due to research equipment and continuous lab ventilation.
>   - Hostel profile has lower commuter travel (residents live on campus) but highest per-capita solid waste generation (6.5 kg/student-month) from cafeteria operations.


---
## 5. Primary Activity Trends: Electricity, Commute, and Solid Waste

### Analysis Questions:
- What are the underlying seasonal rhythms in raw consumption units (kWh, km, kg)?
- Do activity rhythms align with institutional academic and fiscal calendars?

![Figure 5: Activity Time Series & Seasonality Multi-panel](../reports/figures/05_activity_trends_seasonality.png)


In [ ]:
monthly_activity_stats = emissions_df.groupby('month').agg({
    'electricity_kwh': ['mean', 'min', 'max'],
    'travel_km': ['mean', 'min', 'max'],
    'waste_kg': ['mean', 'min', 'max'],
    'procurement_inr': ['mean', 'min', 'max']
}).round(1)

monthly_activity_stats


> [!NOTE]
> **Observations vs. Assumptions (Activity Metric Trends):**
> - **Empirical Observations:**
>   - **Electricity (kWh):** Reaches maximum monthly averages in April (30,347 kWh/bldg) and August (30,004 kWh/bldg). Minimum occurs in December (16,437 kWh/bldg).
>   - **Commuting (km):** Peak commuter travel occurs in October (30,260 km/bldg) and August (29,974 km/bldg); sharp drop in June (6,664 km/bldg, a 78% drop).
>   - **Waste (kg):** Bimodal surges in February (3,502 kg/bldg) and October (3,494 kg/bldg).
>   - **Procurement (INR):** Notable spending spikes in March (₹740,776/bldg average) and July (₹683,522/bldg average).
> - **Underlying Parameterized Assumptions:**
>   - March procurement surge reflects Indian fiscal year closeout budget utilization.
>   - July procurement surge reflects new academic year commencement (laboratory consumables, library books, orientation materials).
>   - Waste spikes in February and October reflect modeled campus festival events.


---
## 6. Per-Student Carbon Intensity Analysis

### Analysis Questions:
- What is the overall carbon intensity per student across the campus?
- How does carbon intensity vary between specialized laboratory facilities and general lecture blocks?

![Figure 6: Per-Student Carbon Intensity Distribution](../reports/figures/06_per_student_emissions.png)


In [ ]:
# Overall campus emissions per student-month
overall_student_months = emissions_df['student_count'].sum()
campus_intensity = emissions_df['total_emissions_kg'].sum() / overall_student_months

print(f"Overall Campus Emissions Intensity: {campus_intensity:.2f} kgCO2e per student-month")

# Building level intensity
bldg_intensity = emissions_df.groupby('building').apply(
    lambda g: g['total_emissions_kg'].sum() / g['student_count'].sum(),
    include_groups=False
).round(2)

print("\nCarbon Intensity by Facility (kgCO2e / student-month):")
print(bldg_intensity.sort_values(ascending=False))


> [!NOTE]
> **Observations vs. Assumptions (Per-Student Intensity):**
> - **Empirical Observations:**
>   - The overall campus-wide intensity is **25.48 kgCO2e per student-month**.
>   - **Administrative Headquarters** exhibits an apparent high ratio (~54.0 kgCO2e/student-month) because student presence in administrative offices is low (staff-dominated operations).
>   - **Science & Engineering Complex** has an intensity of **32.83 kgCO2e/student-month**.
>   - **Academic Block - Arts & Humanities** has the lowest intensity: **15.29 kgCO2e/student-month**.
> - **Underlying Parameterized Assumptions:**
>   - Arts & Humanities has high classroom density (base 1,400 students) and low baseline equipment, whereas Science & Engineering supports intensive laboratory equipment over a smaller student cohort.


---
## 7. Synthesis & Decarbonization Implications

| Dimension | Empirical Observation | Modeling Assumption | Strategic Decarbonization Implication |
| :--- | :--- | :--- | :--- |
| **Primary Category** | Electricity accounts for 79.8% of emissions | Grid emission factor = 0.716 kgCO2e/kWh (CEA 2024) | On-site solar PV and energy efficiency offer the highest carbon reduction leverage per dollar. |
| **Secondary Category** | Travel accounts for 14.2% of emissions | Commute factor = 0.140 kgCO2e/km (DEFRA 2023) | EV campus transit shuttles and hybrid/remote academic scheduling offer moderate reduction. |
| **Facility Priority** | Science Complex + Hostels = 53.3% of total | High equipment load (Science) + 24/7 dining/living (Hostels) | Interventions should prioritize lab HVAC optimization and kitchen waste recovery. |
| **Temporal Seasonality** | Recurring April/August peaks, June/Dec troughs | Summer cooling index (1.45) + semester attendance cycles | Demand management during peak cooling months will yield substantial emissions savings. |

---
*Report generated for BDS-36 Academic Analytics prototype. All activity data synthetic; all emission factors externally documented.*
